# IRIS 27 — GPU reproduction run

Rebuilds the whole corpus and retrains every model on this notebook's GPU.

**Before you start:** upload `iris27_project.zip` (produced on your machine in `dist/`)
when the fourth cell asks for it. Nothing else is needed — the datasets are downloaded
inside Colab.

Runtime → Change runtime type → **GPU** (T4 is enough).

Expected wall clock on a T4: ~25 minutes total.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit('Enable a GPU runtime: Runtime > Change runtime type > GPU')

In [ ]:
%pip install -q spandrel spandrel_extra_arches scikit-learn pandas matplotlib tabulate
import spandrel, spandrel_extra_arches
print('spandrel', spandrel.__version__)

In [ ]:
from google.colab import files
import zipfile, os, shutil

shutil.rmtree('/content/project', ignore_errors=True)
print('Choose iris27_project.zip from dist/')
uploaded = files.upload()
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/project')
os.chdir('/content/project')
print(sorted(os.listdir('.')))
print(sorted(os.listdir('scripts'))[:8], '...')

In [ ]:
# Full pipeline: data -> restoration -> manifest -> probe -> fine-tune -> protocols -> figures
!python notebooks/colab_run.py --stage all --epochs 8

In [ ]:
import shutil, os
shutil.make_archive('/content/iris27_results', 'zip', '/content/project', 'results')
shutil.make_archive('/content/iris27_manifest', 'zip', '/content/project', 'data/manifest.csv')
from google.colab import files
files.download('/content/iris27_results.zip')
files.download('/content/iris27_manifest.zip')